# PyTorch 核心 API 全解

对应 Keras 版本：`Keras应用/第三章-Keras和TensorFlow入门/keras_核心API_全解.ipynb`

**学习路线：**

| 阶段 | 章节 | 内容 | 关键词 |
|------|------|------|--------|
| **一、模型构建** | §1-§3 | nn.Module、nn.Sequential、forward() | Module, Sequential, forward |
| **二、训练流程** | §4-§6 | DataLoader、手动训练、评估 | DataLoader, backward, eval |
| **三、工程实践** | §7-§8 | 保存加载、回调模式 | save, load, callback |

> 全部使用合成数据演示，跑通后替换成自己的数据即可。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")

---
# 第一阶段：模型构建

对应 Keras 版本的层（Layer）与模型（Model）概念。

## §1 层与模型的关系

**层（nn.Module）** = 数据处理模块，内部有可训练参数。

**模型（nn.Module 的子类）** = 多个层组合在一起的结构。

| 概念 | Keras | PyTorch |
|------|-------|---------|
| 层 | `keras.layers.Layer` | `nn.Module` |
| 模型 | `keras.Model` | `nn.Module`（模型本身也是 Module） |
| 参数 | `layer.weights` | `module.parameters()` |
| 前向传播 | `call()` | `forward()` |

In [ ]:
# 查看常见层的参数
linear = nn.Linear(784, 512)
print(f"Linear 层参数：")
for name, param in linear.named_parameters():
    print(f"  {name}: shape={tuple(param.shape)}")

conv = nn.Conv2d(3, 64, kernel_size=3, padding=1)
print(f"\nConv2d 层参数：")
for name, param in conv.named_parameters():
    print(f"  {name}: shape={tuple(param.shape)}")

---
## §2 自定义层

对应 Keras 版本的自定义 Dense 层。

In [ ]:
class SimpleDense(nn.Module):
    """
    自定义全连接层：output = activation(x @ W + b)
    
    对应 Keras 版本的 SimpleDense 层。
    PyTorch 中在 __init__ 中定义参数，在 forward() 中计算。
    """
    
    def __init__(self, in_features, out_features, activation=None):
        super().__init__()
        # 在 __init__ 中直接定义参数
        self.W = nn.Parameter(torch.randn(in_features, out_features) * 0.01)
        self.b = nn.Parameter(torch.zeros(out_features))
        self.activation = activation
    
    def forward(self, x):
        y = x @ self.W + self.b
        if self.activation is not None:
            y = self.activation(y)
        return y

# 测试
my_dense = SimpleDense(784, 32, activation=torch.relu)
x = torch.randn(4, 784)
out = my_dense(x)
print(f"输入: {x.shape} → 输出: {out.shape}")
print(f"参数数量: {sum(p.numel() for p in my_dense.parameters())}")

---
## §3 Sequential vs 自定义 nn.Module

对应 Keras 版本的 Sequential vs 函数式 API。

In [ ]:
# 方式一：nn.Sequential（最简单）
sequential_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 64),
    nn.ReLU(),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 10)
)
print("=== nn.Sequential ===")
print(sequential_model)

In [ ]:
# 方式二：自定义 nn.Module（灵活）

class MLP(nn.Module):
    """
    多层感知机。
    
    与 Keras 函数式 API 的区别：
    - 所有逻辑在 forward() 中显式写出
    - 可以加入任意 Python 控制流（if/for/while）
    """
    
    def __init__(self, input_dim=784, hidden_dim=64, num_classes=10):
        super().__init__()
        
        # 在 __init__ 中定义所有子层
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(x.size(0), -1)  # 展平
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)  # 输出 logits（不带 softmax）
        return x

model = MLP()
print("=== 自定义 nn.Module ===")
print(model)

# 测试前向传播
x = torch.randn(32, 1, 28, 28)  # 模拟一个 batch
out = model(x)
print(f"\n输入: {x.shape} → 输出: {out.shape}")

In [ ]:
# 查看模型参数
print(f"总参数: {sum(p.numel() for p in model.parameters())}")
print(f"可训练参数: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")
print("\n参数详情：")
for name, param in model.named_parameters():
    print(f"  {name}: shape={tuple(param.shape)}")

### 第一阶段小结

| 概念 | 要点 |
|------|------|
| nn.Module | 所有层和模型都继承它 |
| `__init__` | 定义所有子层（"声明零件"） |
| `forward()` | 定义前向传播逻辑（"组装流程"） |
| nn.Sequential | 简单线性堆叠，一行搞定 |
| 自定义 Module | 灵活，可加入任意控制流 |
| 参数访问 | `model.parameters()` / `model.named_parameters()` |

---
# 第二阶段：训练流程

对应 Keras 版本的 compile → fit → evaluate → predict。

## §4 数据加载：DataLoader

对应 Keras 的 `tf.data.Dataset` 和 `batch_size` 参数。

In [ ]:
# 4.1 使用合成数据演示 DataLoader
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)

# 生成模拟二分类数据
N, D = 2000, 20
X = torch.randn(N, D)
y = torch.randint(0, 2, (N,)).long()

# 划分训练/验证集
val_n = 400
train_X, val_X = X[val_n:], X[:val_n]
train_y, val_y = y[val_n:], y[:val_n]

# 创建 Dataset
train_ds = TensorDataset(train_X, train_y)
val_ds = TensorDataset(val_X, val_y)

# 创建 DataLoader（等价于 Keras 的 batch_size + shuffle）
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

print(f"训练集: {len(train_ds)} 样本, {len(train_loader)} 个 batch")
print(f"验证集: {len(val_ds)} 样本, {len(val_loader)} 个 batch")

# 查看一个 batch
for bx, by in train_loader:
    print(f"\n一个 batch: x={bx.shape}, y={by.shape}")
    break

## §5 手动训练循环

对应 Keras 版本的 `model.compile()` + `model.fit()`。

**Keras**：`compile()` + `fit()` 内部自动完成所有步骤。
**PyTorch**：手动写出每一步，但更透明、更灵活。

In [ ]:
# 5.1 构建模型、优化器、损失函数（对应 Keras 的 compile）

model = nn.Sequential(
    nn.Linear(20, 32),
    nn.ReLU(),
    nn.Linear(32, 32),
    nn.ReLU(),
    nn.Linear(32, 2)  # 输出 2 类 logits
)

# 优化器（对应 compile 的 optimizer）
optimizer = optim.RMSprop(model.parameters(), lr=1e-3)

# 损失函数（对应 compile 的 loss）
loss_fn = nn.CrossEntropyLoss()

print(f"模型:\n{model}")
print(f"\n优化器: {optimizer}")
print(f"损失函数: {loss_fn}")

In [ ]:
# 5.2 训练一个 epoch 的函数

def train_epoch(model, loader, optimizer, loss_fn):
    """训练一个 epoch"""
    model.train()  # 设置训练模式
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_x, batch_y in loader:
        # 1. 清零梯度
        optimizer.zero_grad()
        
        # 2. 前向传播
        outputs = model(batch_x)
        
        # 3. 计算损失
        loss = loss_fn(outputs, batch_y)
        
        # 4. 反向传播
        loss.backward()
        
        # 5. 更新权重
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)
    
    return total_loss / len(loader), correct / total


@torch.no_grad()
def evaluate(model, loader, loss_fn):
    """验证模型（对应 Keras 的 evaluate）"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_x, batch_y in loader:
        outputs = model(batch_x)
        loss = loss_fn(outputs, batch_y)
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)
    
    return total_loss / len(loader), correct / total

In [ ]:
# 5.3 完整训练（对应 Keras 的 fit）

epochs = 10
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, loss_fn)
    val_loss, val_acc = evaluate(model, val_loader, loss_fn)
    
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    
    print(f"Epoch {epoch+1:2d}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, "
          f"val_acc={val_acc:.4f}")

In [ ]:
# 5.4 预测（对应 Keras 的 predict）

@torch.no_grad()
def predict_samples(model, loader, num=5):
    model.eval()
    x, y = next(iter(loader))
    outputs = model(x[:num])
    probs = torch.softmax(outputs, dim=1)
    preds = torch.argmax(probs, dim=1)
    
    print(f"{'样本':>6} | {'预测':>6} | {'概率':>8} | {'真实':>6}")
    print("-" * 40)
    for i in range(num):
        match = "✓" if preds[i] == y[i] else "✗"
        print(f"  #{i:>4} | {preds[i]:>6} | {probs[i, preds[i]].item():>8.4f} | {y[i]:>6} {match}")

predict_samples(model, val_loader)

---
# 第三阶段：工程实践

对应 Keras 版本的模型保存、回调。

## §7 模型保存与加载

对应 Keras 的 `model.save()` 和 `keras.models.load_model()`。

In [ ]:
# 7.1 保存模型状态（推荐方式）
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': epochs,
    'train_loss': history['train_loss'][-1],
}, 'demo_checkpoint.pt')

print("模型已保存到 demo_checkpoint.pt")

# 加载模型
checkpoint = torch.load('demo_checkpoint.pt', weights_only=False)

# 恢复模型状态
new_model = nn.Sequential(
    nn.Linear(20, 32),
    nn.ReLU(),
    nn.Linear(32, 32),
    nn.ReLU(),
    nn.Linear(32, 2)
)
new_model.load_state_dict(checkpoint['model_state_dict'])

# 验证
x_test = torch.randn(5, 20)
pred_orig = model(x_test)
pred_loaded = new_model(x_test)
print(f"预测结果一致: {torch.allclose(pred_orig, pred_loaded)}")

# 清理
import os
os.remove('demo_checkpoint.pt')

## §8 回调模式

对应 Keras 版本的 `EarlyStopping`、`ModelCheckpoint`、`ReduceLROnPlateau`。

In [ ]:
# 8.1 回调基类

class Callback:
    """回调函数基类"""
    def on_epoch_end(self, epoch, logs=None): pass


class EarlyStopping(Callback):
    """
    早停回调。
    对应 Keras 的 keras.callbacks.EarlyStopping
    """
    def __init__(self, monitor="val_loss", patience=3, mode="min"):
        self.monitor = monitor
        self.patience = patience
        self.mode = mode
        self.counter = 0
        self.best_value = None
        self.should_stop = False
    
    def on_epoch_end(self, epoch, logs=None):
        current = logs.get(self.monitor)
        if current is None:
            return
        
        if self.best_value is None:
            self.best_value = current
        elif (self.mode == "min" and current < self.best_value) or \
             (self.mode == "max" and current > self.best_value):
            self.best_value = current
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                print(f"  [EarlyStopping] {self.monitor} 在 {self.patience} 轮内未改善")


class ModelCheckpoint(Callback):
    """
    模型检查点。
    对应 Keras 的 keras.callbacks.ModelCheckpoint
    """
    def __init__(self, filepath, monitor="val_loss", mode="min"):
        self.filepath = filepath
        self.monitor = monitor
        self.mode = mode
        self.best_value = None
    
    def on_epoch_end(self, epoch, logs=None):
        current = logs.get(self.monitor)
        if current is None:
            return
        
        if self.best_value is None or \
           (self.mode == "min" and current < self.best_value) or \
           (self.mode == "max" and current > self.best_value):
            self.best_value = current
            torch.save(logs['model'].state_dict(), self.filepath)
            print(f"  [Checkpoint] 保存到 {self.filepath}")


print("回调类定义完成")

In [ ]:
# 8.2 使用回调训练

def train_with_callbacks(model, optimizer, loss_fn, train_loader, val_loader,
                         epochs, callbacks):
    """支持回调的训练循环"""
    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, loss_fn)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn)
        
        logs = {
            "train_loss": train_loss, "train_acc": train_acc,
            "val_loss": val_loss, "val_acc": val_acc,
            "model": model
        }
        
        print(f"Epoch {epoch+1:2d}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")
        
        for cb in callbacks:
            cb.on_epoch_end(epoch, logs)
        
        if any(cb.should_stop for cb in callbacks if hasattr(cb, 'should_stop')):
            print(f"训练在第 {epoch+1} 轮提前结束")
            break

# 使用示例
model2 = nn.Sequential(
    nn.Linear(20, 32), nn.ReLU(),
    nn.Linear(32, 32), nn.ReLU(),
    nn.Linear(32, 2)
)
optimizer2 = optim.RMSprop(model2.parameters(), lr=1e-3)
loss_fn2 = nn.CrossEntropyLoss()

callbacks = [
    EarlyStopping(monitor="val_loss", patience=3, mode="min"),
    ModelCheckpoint(filepath="best_model.pt", monitor="val_loss", mode="min")
]

train_with_callbacks(model2, optimizer2, loss_fn2, train_loader, val_loader,
                     epochs=20, callbacks=callbacks)

# 清理
import os
if os.path.exists("best_model.pt"):
    os.remove("best_model.pt")

---
# 总结

## Keras vs PyTorch 核心对照

| 步骤 | Keras | PyTorch |
|------|-------|---------|
| 定义模型 | `Sequential([...])` / 函数式 API | `nn.Sequential(...)` / `nn.Module` 子类 |
| 配置训练 | `compile(optimizer, loss, metrics)` | 创建 `optimizer` + `loss_fn` |
| 数据管道 | `tf.data.Dataset` | `DataLoader` + `TensorDataset` |
| 训练 | `fit(x, y, epochs, batch_size)` | 手写 `for` 循环 |
| 评估 | `evaluate(x, y)` | 手写 `torch.no_grad()` 循环 |
| 预测 | `predict(x)` | `model(x)` |
| 保存 | `model.save()` | `torch.save(model.state_dict(), ...)` |
| 加载 | `keras.models.load_model()` | `model.load_state_dict(torch.load(...))` |
| 回调 | `keras.callbacks.*` | 自定义 `Callback` 类 |

## PyTorch 训练六步曲

```
1. 定义模型: model = nn.Sequential(...) / nn.Module 子类
2. 定义优化器: optimizer = optim.Adam(model.parameters())
3. 定义损失: loss_fn = nn.CrossEntropyLoss()
4. 数据加载: loader = DataLoader(dataset, batch_size, shuffle)
5. 训练循环:
   model.train()
   optimizer.zero_grad()
   outputs = model(batch_x)
   loss = loss_fn(outputs, batch_y)
   loss.backward()
   optimizer.step()
6. 验证循环:
   model.eval() + torch.no_grad()
```

---
**下一步学习**：打开第四章 notebooks 学习真实数据集上的分类和回归任务。